In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Enable inline plotting
get_ipython().run_line_magic('matplotlib', 'inline')

# Load the dataset
df = pd.read_csv('product_quality_manufacturing_analytics.csv')
df.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['Product_Quality'].value_counts()

In [ ]:
df['Product_Quality'].value_counts(normalize=True) * 100

In [ ]:
# Check missing values
print('Missing Values:\n', df.isnull().sum())

# Check duplicates
print('Duplicate count:', df.duplicated().sum())

In [ ]:
# Remove Duplicates
df = df.drop_duplicates()
print('New dataset shape after removing duplicates:', df.shape)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Product_Quality', data=df, palette='viridis')
plt.title('Product Quality Distribution (0 = Good, 1 = Defective)')
plt.xlabel('Product Quality')
plt.ylabel('Count')
plt.show()

# Insight: The dataset is heavily imbalanced — 84% Defective (2723 samples) vs
# 16% Good (517 samples). This means accuracy alone is a misleading metric.
# A model predicting "Defective" every time would still score 84% accuracy.
# We will use F1-Score and ROC-AUC for proper evaluation.

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()

# Insight: The three strongest predictors of Product_Quality are
# Equipment_Maintenance_Count (0.30), Defect_Rate (0.25), and Quality_Score (-0.20).
# Higher maintenance count and defect rate are associated with defective products,
# while higher quality scores are associated with good products.

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x='Product_Quality', y='Defect_Rate', data=df, palette='Set2')
plt.title('Defect Rate by Product Quality')
plt.xlabel('Product Quality')
plt.ylabel('Defect Rate')
plt.show()

# Insight: Defective products have a noticeably higher average defect rate (~2.89)
# compared to good products (~2.01). This makes business sense — a rising defect
# rate during production is an early warning signal for quality failure.

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x='Product_Quality', y='Quality_Score', data=df, palette='Set2')
plt.title('Quality Score by Product Quality')
plt.xlabel('Product Quality')
plt.ylabel('Quality Score')
plt.show()

# Insight: Good products have a higher average Quality Score (~85.4) compared to
# defective ones (~79.1). Quality Score is a strong differentiator and could be
# used as a real-time threshold alert in a production monitoring system.

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x='Product_Quality', y='Machine_Downtime_Hours', data=df, palette='Set2')
plt.title('Machine Downtime Hours by Product Quality')
plt.xlabel('Product Quality')
plt.ylabel('Machine Downtime Hours')
plt.show()

# Insight: Machine downtime shows almost no difference between good (~2.49 hrs)
# and defective (~2.50 hrs) products. This tells us downtime alone is not a
# reliable predictor — other factors like defect rate and quality score matter more.

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x='Product_Quality', y='Energy_Consumption', data=df, palette='Set2')
plt.title('Energy Consumption by Product Quality')
plt.xlabel('Product Quality')
plt.ylabel('Energy Consumption')
plt.show()

# Insight: Energy consumption is nearly identical for good (~2975 kWh) and
# defective (~2991 kWh) products. Energy is not a meaningful quality predictor
# in this dataset, which is useful to know for feature selection.

In [ ]:
X = df.drop('Product_Quality', axis=1)
y = df['Product_Quality']

# ─────────────────────────────────────────────────────────────────
# CELL 18 - Train Test Split
# FIX: Added stratify=y to preserve class ratio in both splits
# ─────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # ensures same 84/16 ratio in train and test
)
print('Train size:', X_train.shape)
print('Test size:', X_test.shape)

# ─────────────────────────────────────────────────────────────────
# CELL 19 - Scale features for Logistic Regression
# FIX: Added StandardScaler — LR is sensitive to feature scale
# ─────────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit only on train
X_test_sc  = scaler.transform(X_test)         # transform test with same scaler

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)  # uses scaled features

# ─────────────────────────────────────────────────────────────────
# CELL 21 - Random Forest
# FIX: Added class_weight='balanced' to handle 84/16 class imbalance
# ─────────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'  # penalises misclassifying the minority class
)
rf.fit(X_train, y_train)

In [ ]:
y_pred_lr = lr.predict(X_test_sc)  # uses scaled test features
y_pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_lr))
print('Random Forest Accuracy:', accuracy_score(y_test, y_pred_rf))

# Note: Accuracy is not the best metric here due to class imbalance.
# Focus on F1-Score and ROC-AUC in the sections below.

In [ ]:
from sklearn.metrics import classification_report

print('Random Forest Classification Report:\n')
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_rf)
print('Confusion Matrix:\n', cm)

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Good', 'Defective'],
            yticklabels=['Good', 'Defective'])
plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

y_prob = rf.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_prob)
print('Random Forest ROC-AUC Score:', auc_score)

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {auc_score:.4f})', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
})
feature_importance.sort_values(by='Importance', ascending=False, inplace=True)
feature_importance.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance.head(10), palette='viridis')
plt.title('Top 10 Most Important Features - Random Forest')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

# Insight: Equipment_Maintenance_Count, Defect_Rate, and Quality_Score are the top
# 3 features driving predictions. Business recommendation: monitor these three KPIs
# in real time on the production floor to catch defects before products are finished.

In [ ]:
import pickle
import os

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save the Random Forest Model
model_path = 'models/random_forest_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(rf, f)
print(f'Model successfully saved to {model_path}')

In [ ]:
with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

# Make a prediction on the first sample of the test set
dummy_pred = loaded_model.predict(X_test.iloc[[0]])
print('Test prediction class:', dummy_pred[0])
print('Actual class:', y_test.iloc[0])